In [32]:
"""
Investment Banking Valuation Toolkit with Full Sensitivity Analysis and Excel Export

This script calculates the equity value of a given public company using five
different valuation methodologies, generates sensitivity tables and heatmaps,
and exports all results to an Excel workbook with a football field chart.

Author: Aarsh

"""

'\nInvestment Banking Valuation Toolkit with Full Sensitivity Analysis and Excel Export\n\nThis script calculates the equity value of a given public company using five\ndifferent valuation methodologies, generates sensitivity tables and heatmaps,\nand exports all results to an Excel workbook with a football field chart.\n\nAuthor: Aarsh\n\n'

In [33]:
import pandas as pd
import numpy as np
import requests
import yfinance as yf
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# For Excel export
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl.utils.dataframe import dataframe_to_rows

In [34]:
# ============================================================================
# SECTION 1: DATA FETCHING FUNCTIONS
# ============================================================================

def get_company_info(ticker):
    """Fetch basic company information and financial statements."""
    stock = yf.Ticker(ticker)
    info = stock.info
    income_stmt = stock.financials
    balance_sheet = stock.balance_sheet
    cash_flow = stock.cashflow
    hist = stock.history(period="1y")
    current_price = hist['Close'].iloc[-1]
    shares_outstanding = info.get('sharesOutstanding', info.get('sharesOutstanding', 0))
    
    return {
        'name': info.get('longName', ticker),
        'sector': info.get('sector', 'Unknown'),
        'industry': info.get('industry', 'Unknown'),
        'current_price': current_price,
        'shares_outstanding': shares_outstanding,
        'market_cap': current_price * shares_outstanding,
        'income_stmt': income_stmt,
        'balance_sheet': balance_sheet,
        'cash_flow': cash_flow,
        'info': info
    }


def get_peers(ticker, sector, num_peers=10):
    """Identify comparable public companies based on sector and market cap."""
    # 1. Download the page with a browser-like User-Agent header
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    
    # 2. Parse the HTML content to extract tables
    sp500 = pd.read_html(response.text)[0]  # The first table on the page

    sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
    sector_peers = sp500[sp500['GICS Sector'] == sector]['Symbol'].tolist()
    
    peer_data = []
    for sym in sector_peers[:50]:
        try:
            stock = yf.Ticker(sym)
            info = stock.info
            market_cap = info.get('marketCap', 0)
            if market_cap > 0:
                peer_data.append({'symbol': sym, 'name': info.get('longName', sym), 'market_cap': market_cap})
        except:
            continue
    peer_data = sorted(peer_data, key=lambda x: x['market_cap'])
    target_mcap = get_company_info(ticker)['market_cap']
    similar_peers = [p for p in peer_data if 0.5 * target_mcap <= p['market_cap'] <= 2 * target_mcap]
    return similar_peers[:num_peers]

def get_peer_financials(peer_symbol):
    """Fetch financial data for a peer company."""
    try:
        stock = yf.Ticker(peer_symbol)
        info = stock.info
        income_stmt = stock.financials
        balance_sheet = stock.balance_sheet
        if income_stmt.empty or balance_sheet.empty:
            return None
        revenue = income_stmt.loc['Total Revenue'].iloc[0] if 'Total Revenue' in income_stmt.index else None
        ebitda = income_stmt.loc['EBITDA'].iloc[0] if 'EBITDA' in income_stmt.index else None
        net_income = income_stmt.loc['Net Income'].iloc[0] if 'Net Income' in income_stmt.index else None
        total_debt = balance_sheet.loc['Total Debt'].iloc[0] if 'Total Debt' in balance_sheet.index else 0
        cash = balance_sheet.loc['Cash And Cash Equivalents'].iloc[0] if 'Cash And Cash Equivalents' in balance_sheet.index else 0
        market_cap = info.get('marketCap', 0)
        enterprise_value = market_cap + total_debt - cash
        return {
            'symbol': peer_symbol, 'revenue': revenue, 'ebitda': ebitda, 'net_income': net_income,
            'market_cap': market_cap, 'enterprise_value': enterprise_value, 'total_debt': total_debt, 'cash': cash
        }
    except:
        return None

In [35]:
# ============================================================================
# SECTION 2: VALUATION METHODOLOGIES (with sensitivity)
# ============================================================================

class DCFValuation:
    def __init__(self, company_data):
        self.company_data = company_data
        self.wacc = 0.08
        self.perpetual_growth = 0.02
    
    def calculate_free_cash_flow(self):
        cf = self.company_data['cash_flow']
        if cf.empty:
            return None
        try:
            ocf = cf.loc['Operating Cash Flow'].iloc[0] if 'Operating Cash Flow' in cf.index else None
            capex = cf.loc['Capital Expenditure'].iloc[0] if 'Capital Expenditure' in cf.index else None
            if ocf and capex:
                return ocf + capex  # capex is negative
        except:
            pass
        return None
    
    def project_fcf(self, historical_fcf, growth_rates):
        projected = []
        current = historical_fcf
        for rate in growth_rates:
            current *= (1 + rate)
            projected.append(current)
        return projected
    
    def calculate(self, growth_scenarios=None):
        historical_fcf = self.calculate_free_cash_flow()
        if historical_fcf is None:
            return None
        if growth_scenarios is None:
            growth_scenarios = {
                'conservative': [0.01, 0.02, 0.02, 0.01, 0.01],
                'base': [0.03, 0.04, 0.04, 0.03, 0.03],
                'aggressive': [0.05, 0.06, 0.06, 0.05, 0.05]
            }
        valuations = []
        for scenario, rates in growth_scenarios.items():
            proj_fcf = self.project_fcf(historical_fcf, rates)
            pv_fcf = sum([fcf / (1 + self.wacc)**(t+1) for t, fcf in enumerate(proj_fcf)])
            tv = proj_fcf[-1] * (1 + self.perpetual_growth) / (self.wacc - self.perpetual_growth)
            pv_tv = tv / (1 + self.wacc)**len(proj_fcf)
            ev = pv_fcf + pv_tv
            debt = self.company_data['balance_sheet'].loc['Total Debt'].iloc[0] if 'Total Debt' in self.company_data['balance_sheet'].index else 0
            cash = self.company_data['balance_sheet'].loc['Cash And Cash Equivalents'].iloc[0] if 'Cash And Cash Equivalents' in self.company_data['balance_sheet'].index else 0
            equity = ev - debt + cash
            per_share = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding'] > 0 else 0
            valuations.append(per_share)
        return min(valuations), max(valuations)
    
    def sensitivity_table(self, wacc_range, growth_range):
        """Return DataFrame of value per share for each (wacc, terminal_growth)"""
        historical_fcf = self.calculate_free_cash_flow()
        if historical_fcf is None:
            return None
        base_growth = [0.03, 0.04, 0.04, 0.03, 0.03]
        proj_fcf = self.project_fcf(historical_fcf, base_growth)
        results = []
        for wacc in wacc_range:
            row = []
            for term_growth in growth_range:
                tv = proj_fcf[-1] * (1 + term_growth) / (wacc - term_growth)
                pv_tv = tv / (1 + wacc)**len(proj_fcf)
                pv_fcf = sum([fcf / (1 + wacc)**(t+1) for t, fcf in enumerate(proj_fcf)])
                ev = pv_fcf + pv_tv
                debt = self.company_data['balance_sheet'].loc['Total Debt'].iloc[0] if 'Total Debt' in self.company_data['balance_sheet'].index else 0
                cash = self.company_data['balance_sheet'].loc['Cash And Cash Equivalents'].iloc[0] if 'Cash And Cash Equivalents' in self.company_data['balance_sheet'].index else 0
                equity = ev - debt + cash
                per_share = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding'] > 0 else 0
                row.append(per_share)
            results.append(row)
        return pd.DataFrame(results, index=wacc_range, columns=growth_range)

class TradingCompsValuation:
    def __init__(self, company_data, peers):
        self.company_data = company_data
        self.peers = peers
    
    def get_target_financials(self):
        inc = self.company_data['income_stmt']
        bal = self.company_data['balance_sheet']
        revenue = inc.loc['Total Revenue'].iloc[0] if 'Total Revenue' in inc.index else None
        ebitda = inc.loc['EBITDA'].iloc[0] if 'EBITDA' in inc.index else None
        net_income = inc.loc['Net Income'].iloc[0] if 'Net Income' in inc.index else None
        debt = bal.loc['Total Debt'].iloc[0] if 'Total Debt' in bal.index else 0
        cash = bal.loc['Cash And Cash Equivalents'].iloc[0] if 'Cash And Cash Equivalents' in bal.index else 0
        return {'revenue': revenue, 'ebitda': ebitda, 'net_income': net_income, 'debt': debt, 'cash': cash}
    
    def calculate(self):
        peer_multiples = {'ev_revenue': [], 'ev_ebitda': [], 'pe': []}
        for peer in self.peers:
            fin = get_peer_financials(peer['symbol'])
            if fin and fin['revenue'] and fin['revenue']>0:
                peer_multiples['ev_revenue'].append(fin['enterprise_value'] / fin['revenue'])
            if fin and fin['ebitda'] and fin['ebitda']>0:
                peer_multiples['ev_ebitda'].append(fin['enterprise_value'] / fin['ebitda'])
            if fin and fin['net_income'] and fin['net_income']>0:
                peer_multiples['pe'].append(fin['market_cap'] / fin['net_income'])
        if not any(peer_multiples.values()):
            return None
        # Calculate percentiles
        percentiles = {}
        for k, v in peer_multiples.items():
            if v:
                percentiles[k] = {
                    '25th': np.percentile(v, 25),
                    '50th': np.percentile(v, 50),
                    '75th': np.percentile(v, 75)
                }
        target = self.get_target_financials()
        valuations = []
        for mult_type, pcts in percentiles.items():
            for pct_name, mult in pcts.items():
                if mult_type == 'ev_revenue' and target['revenue']:
                    ev = target['revenue'] * mult
                    equity = ev - target['debt'] + target['cash']
                    price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                    valuations.append(price)
                elif mult_type == 'ev_ebitda' and target['ebitda']:
                    ev = target['ebitda'] * mult
                    equity = ev - target['debt'] + target['cash']
                    price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                    valuations.append(price)
                elif mult_type == 'pe' and target['net_income']:
                    equity = target['net_income'] * mult
                    price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                    valuations.append(price)
        if valuations:
            return min(valuations), max(valuations)
        return None
    
    def get_percentile_valuations(self):
        """Return a dict of valuations at 25th, 50th, 75th percentiles for each multiple"""
        peer_multiples = {'ev_revenue': [], 'ev_ebitda': [], 'pe': []}
        for peer in self.peers:
            fin = get_peer_financials(peer['symbol'])
            if fin and fin['revenue'] and fin['revenue']>0:
                peer_multiples['ev_revenue'].append(fin['enterprise_value'] / fin['revenue'])
            if fin and fin['ebitda'] and fin['ebitda']>0:
                peer_multiples['ev_ebitda'].append(fin['enterprise_value'] / fin['ebitda'])
            if fin and fin['net_income'] and fin['net_income']>0:
                peer_multiples['pe'].append(fin['market_cap'] / fin['net_income'])
        target = self.get_target_financials()
        result = {}
        for mult_type, v in peer_multiples.items():
            if v:
                p25 = np.percentile(v, 25)
                p50 = np.percentile(v, 50)
                p75 = np.percentile(v, 75)
                for label, mult in [('25th', p25), ('50th', p50), ('75th', p75)]:
                    if mult_type == 'ev_revenue' and target['revenue']:
                        ev = target['revenue'] * mult
                        equity = ev - target['debt'] + target['cash']
                        price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                        result[f'{mult_type}_{label}'] = price
                    elif mult_type == 'ev_ebitda' and target['ebitda']:
                        ev = target['ebitda'] * mult
                        equity = ev - target['debt'] + target['cash']
                        price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                        result[f'{mult_type}_{label}'] = price
                    elif mult_type == 'pe' and target['net_income']:
                        equity = target['net_income'] * mult
                        price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                        result[f'{mult_type}_{label}'] = price
        return result

class PrecedentTransactionValuation:
    def __init__(self, company_data):
        self.company_data = company_data
        # Example transaction multiples (EV/Revenue and EV/EBITDA)
        self.transaction_multiples = {
            'ev_revenue': [1.5, 2.0, 2.5, 3.0, 3.5],
            'ev_ebitda': [8.0, 10.0, 12.0, 14.0, 16.0]
        }
    
    def get_target_financials(self):
        inc = self.company_data['income_stmt']
        bal = self.company_data['balance_sheet']
        revenue = inc.loc['Total Revenue'].iloc[0] if 'Total Revenue' in inc.index else None
        ebitda = inc.loc['EBITDA'].iloc[0] if 'EBITDA' in inc.index else None
        debt = bal.loc['Total Debt'].iloc[0] if 'Total Debt' in bal.index else 0
        cash = bal.loc['Cash And Cash Equivalents'].iloc[0] if 'Cash And Cash Equivalents' in bal.index else 0
        return {'revenue': revenue, 'ebitda': ebitda, 'debt': debt, 'cash': cash}
    
    def calculate(self):
        target = self.get_target_financials()
        valuations = []
        if target['revenue']:
            for mult in self.transaction_multiples['ev_revenue']:
                ev = target['revenue'] * mult
                equity = ev - target['debt'] + target['cash']
                price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                valuations.append(price)
        if target['ebitda']:
            for mult in self.transaction_multiples['ev_ebitda']:
                ev = target['ebitda'] * mult
                equity = ev - target['debt'] + target['cash']
                price = equity / self.company_data['shares_outstanding'] if self.company_data['shares_outstanding']>0 else 0
                valuations.append(price)
        if valuations:
            return min(valuations), max(valuations)
        return None

class LBOValuation:
    def __init__(self, company_data):
        self.company_data = company_data
        self.target_irr = 0.20
        self.holding_period = 5
        self.debt_ratio = 0.60
        self.interest_rate = 0.06
        self.exit_multiple = 10.0
    
    def calculate(self):
        inc = self.company_data['income_stmt']
        ebitda = inc.loc['EBITDA'].iloc[0] if 'EBITDA' in inc.index else None
        if ebitda is None:
            return None
        growth = 0.03
        exit_ebitda = ebitda * (1 + growth) ** self.holding_period
        exit_ev = exit_ebitda * self.exit_multiple
        initial_debt = exit_ev * self.debt_ratio
        annual_interest = initial_debt * self.interest_rate
        debt_repayment = initial_debt * 0.20
        debt_at_exit = initial_debt - (debt_repayment * self.holding_period)
        exit_equity = exit_ev - debt_at_exit
        max_entry_equity = exit_equity / (1 + self.target_irr) ** self.holding_period
        shares = self.company_data['shares_outstanding']
        if shares > 0:
            per_share = max_entry_equity / shares
            return per_share * 0.8, per_share * 1.2
        return None
    
    def sensitivity_table(self, purchase_multiple_range, exit_multiple_range):
        """Return DataFrame of entry price for different purchase and exit multiples."""
        inc = self.company_data['income_stmt']
        ebitda = inc.loc['EBITDA'].iloc[0] if 'EBITDA' in inc.index else None
        if ebitda is None:
            return None
        shares = self.company_data['shares_outstanding']
        results = []
        for purch_mult in purchase_multiple_range:
            row = []
            for exit_mult in exit_multiple_range:
                entry_ev = ebitda * purch_mult
                exit_ebitda = ebitda * (1 + 0.03) ** self.holding_period
                exit_ev = exit_ebitda * exit_mult
                initial_debt = entry_ev * self.debt_ratio
                debt_repayment = initial_debt * 0.20
                debt_at_exit = initial_debt - (debt_repayment * self.holding_period)
                exit_equity = exit_ev - debt_at_exit
                max_entry_equity = exit_equity / (1 + self.target_irr) ** self.holding_period
                per_share = max_entry_equity / shares if shares > 0 else 0
                row.append(per_share)
            results.append(row)
        return pd.DataFrame(results, index=purchase_multiple_range, columns=exit_multiple_range)

class SumOfPartsValuation:
    def __init__(self, company_data):
        self.company_data = company_data
        self.segment_multipliers = {
            'Technology': 25.0, 'Financials': 15.0, 'Healthcare': 20.0,
            'Consumer': 18.0, 'Industrial': 12.0
        }
    
    def calculate(self):
        sector = self.company_data['sector']
        inc = self.company_data['income_stmt']
        net_income = inc.loc['Net Income'].iloc[0] if 'Net Income' in inc.index else None
        if net_income is None:
            return None
        multiple = self.segment_multipliers.get(sector, 15.0)
        equity = net_income * multiple
        shares = self.company_data['shares_outstanding']
        if shares > 0:
            per_share = equity / shares
            return per_share * 0.8, per_share * 1.2
        return None

In [36]:
# ============================================================================
# SECTION 3: FOOTBALL FIELD CHART AND EXCEL EXPORT
# ============================================================================

def create_football_field_chart(valuations, company_name, current_price, filename='football_field.png'):
    """Generate football field chart with current price and median valuation lines."""
    methods = list(valuations.keys())
    ranges = list(valuations.values())
    # Calculate midpoints for each method
    midpoints = [(r[0] + r[1]) / 2 for r in ranges]
    # Overall median of midpoints
    median_valuation = np.median(midpoints)
    
    # Sort methods by midpoint value
    sorted_indices = np.argsort(midpoints)
    methods = [methods[i] for i in sorted_indices]
    ranges = [ranges[i] for i in sorted_indices]
    
    fig, ax = plt.subplots(figsize=(12,8))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    y_pos = np.arange(len(methods))
    
    for i, (method, (low, high)) in enumerate(zip(methods, ranges)):
        ax.barh(i, high-low, left=low, height=0.5, color=colors[i%len(colors)], alpha=0.7, edgecolor='black')
        ax.text(low, i, f'${low:.2f}', va='center', ha='right', fontsize=9)
        ax.text(high, i, f'${high:.2f}', va='center', ha='left', fontsize=9)
    
    # Current price line (red dashed)
    ax.axvline(x=current_price, color='red', linestyle='--', linewidth=2, label=f'Current Price: ${current_price:.2f}')
    
    # Median valuation line (green dashed)
    ax.axvline(x=median_valuation, color='green', linestyle='--', linewidth=2, label=f'Median Valuation: ${median_valuation:.2f}')
    
    # Plot midpoints as black dots
    for i, (low, high) in enumerate(ranges):
        ax.plot((low+high)/2, i, 'o', color='black', markersize=6)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(methods)
    ax.set_xlabel('Valuation per Share ($)')
    ax.set_title(f'Football Field Chart: {company_name}')
    ax.legend(loc='lower right')
    ax.grid(True, axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    return filename

def export_to_excel(valuations, company_name, current_price, dcf_sens_df, lbo_sens_df, trading_comps_percentiles, output_file='Valuation_Report.xlsx'):
    """Export all valuation results and sensitivity tables to Excel."""
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Summary sheet
        summary_data = []
        for method, (low, high) in valuations.items():
            summary_data.append({'Method': method, 'Low ($)': low, 'High ($)': high, 'Midpoint ($)': (low+high)/2})
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_excel(writer, sheet_name='Valuation Summary', index=False)
        
        # Current price and stats
        stats_df = pd.DataFrame({
            'Metric': ['Current Price', 'Overall Low', 'Overall High', 'Mean Valuation', 'Median Valuation'],
            'Value ($)': [
                current_price,
                min(v[0] for v in valuations.values()),
                max(v[1] for v in valuations.values()),
                np.mean([(v[0]+v[1])/2 for v in valuations.values()]),
                np.median([(v[0]+v[1])/2 for v in valuations.values()])
            ]
        })
        stats_df.to_excel(writer, sheet_name='Summary Stats', index=False)
        
        # DCF sensitivity table
        if dcf_sens_df is not None:
            dcf_sens_df.to_excel(writer, sheet_name='DCF Sensitivity (WACC vs Term Growth)')
        
        # LBO sensitivity table
        if lbo_sens_df is not None:
            lbo_sens_df.to_excel(writer, sheet_name='LBO Sensitivity (Purchase vs Exit Multiple)')
        
        # Trading Comps percentile valuations
        if trading_comps_percentiles:
            tc_df = pd.DataFrame(list(trading_comps_percentiles.items()), columns=['Multiple', 'Implied Price ($)'])
            tc_df.to_excel(writer, sheet_name='Trading Comps Percentiles', index=False)
        
        # Also include the football field chart image
        try:
            from openpyxl.drawing.image import Image as XLImage
            img_path = 'football_field.png'
            if os.path.exists(img_path):
                img = XLImage(img_path)
                img.width = 800
                img.height = 500
                worksheet = writer.sheets['Valuation Summary']
                worksheet.add_image(img, 'E5')
        except:
            pass
    print(f"Excel report saved as {output_file}")

In [39]:
# ============================================================================
# SECTION 4: MAIN EXECUTION
# ============================================================================

def main():
    print("="*80)
    print("INVESTMENT BANKING VALUATION TOOLKIT (with Full Sensitivity)")
    print("="*80)
    ticker = input("\nEnter company ticker symbol (e.g., AAPL, MSFT, JPM): ").strip().upper()
    print(f"\nAnalyzing {ticker}...")
    
    try:
        company_data = get_company_info(ticker)
        print(f"Company: {company_data['name']}")
        print(f"Sector: {company_data['sector']}")
        print(f"Current Price: ${company_data['current_price']:.2f}")
    except Exception as e:
        print(f"Error fetching data: {e}")
        return
    
    valuations = {}
    
    # 1. DCF
    print("\n--- DCF Valuation ---")
    dcf = DCFValuation(company_data)
    dcf_range = dcf.calculate()
    if dcf_range:
        valuations['Discounted Cash Flow (DCF)'] = dcf_range
        print(f"Range: ${dcf_range[0]:.2f} - ${dcf_range[1]:.2f}")
        # Sensitivity table
        wacc_vals = [0.06, 0.07, 0.08, 0.09, 0.10]
        growth_vals = [0.01, 0.015, 0.02, 0.025, 0.03]
        dcf_sens = dcf.sensitivity_table(wacc_vals, growth_vals)
        if dcf_sens is not None:
            print("DCF sensitivity table generated.")
            # Heatmap
            plt.figure(figsize=(10,6))
            sns.heatmap(dcf_sens, annot=True, fmt='.2f', cmap='RdYlGn', xticklabels=growth_vals, yticklabels=wacc_vals)
            plt.title('DCF Sensitivity: Value per Share vs WACC and Terminal Growth')
            plt.xlabel('Terminal Growth Rate')
            plt.ylabel('WACC')
            plt.tight_layout()
            plt.savefig('dcf_sensitivity_heatmap.png', dpi=150)
            plt.close()
            print("Heatmap saved as dcf_sensitivity_heatmap.png")
    else:
        dcf_sens = None
    
    # 2. Trading Comps
    print("\n--- Comparable Company Analysis ---")
    peers = get_peers(ticker, company_data['sector'])
    print(f"Found {len(peers)} peers")
    if peers:
        trading = TradingCompsValuation(company_data, peers)
        comps_range = trading.calculate()
        if comps_range:
            valuations['Comparable Company Analysis'] = comps_range
            print(f"Range: ${comps_range[0]:.2f} - ${comps_range[1]:.2f}")
            trading_percentiles = trading.get_percentile_valuations()
            print("Percentile valuations computed.")
        else:
            trading_percentiles = None
    else:
        trading_percentiles = None
    
    # 3. Precedent Transactions
    print("\n--- Precedent Transaction Analysis ---")
    precedent = PrecedentTransactionValuation(company_data)
    precedent_range = precedent.calculate()
    if precedent_range:
        valuations['Precedent Transactions'] = precedent_range
        print(f"Range: ${precedent_range[0]:.2f} - ${precedent_range[1]:.2f}")
    
    # 4. LBO
    print("\n--- LBO Analysis ---")
    lbo = LBOValuation(company_data)
    lbo_range = lbo.calculate()
    if lbo_range:
        valuations['Leveraged Buyout (LBO)'] = lbo_range
        print(f"Range: ${lbo_range[0]:.2f} - ${lbo_range[1]:.2f}")
        # LBO sensitivity table
        purch_mult_range = [8, 9, 10, 11, 12]
        exit_mult_range = [8, 9, 10, 11, 12]
        lbo_sens = lbo.sensitivity_table(purch_mult_range, exit_mult_range)
        if lbo_sens is not None:
            print("LBO sensitivity table generated.")
            plt.figure(figsize=(10,6))
            sns.heatmap(lbo_sens, annot=True, fmt='.2f', cmap='RdYlGn', xticklabels=exit_mult_range, yticklabels=purch_mult_range)
            plt.title('LBO Sensitivity: Entry Price vs Purchase Multiple and Exit Multiple')
            plt.xlabel('Exit Multiple')
            plt.ylabel('Purchase Multiple')
            plt.tight_layout()
            plt.savefig('lbo_sensitivity_heatmap.png', dpi=150)
            plt.close()
    else:
        lbo_sens = None
    
    # 5. Sum-of-the-Parts
    print("\n--- Sum-of-the-Parts Analysis ---")
    sotp = SumOfPartsValuation(company_data)
    sotp_range = sotp.calculate()
    if sotp_range:
        valuations['Sum-of-the-Parts (SOTP)'] = sotp_range
        print(f"Range: ${sotp_range[0]:.2f} - ${sotp_range[1]:.2f}")
    
    # Football field chart
    if valuations:
        chart_file = create_football_field_chart(valuations, company_data['name'], company_data['current_price'])
        print(f"\nFootball field chart saved as {chart_file}")
        
        # Export to Excel
        export_to_excel(valuations, company_data['name'], company_data['current_price'],
                        dcf_sens, lbo_sens, trading_percentiles)
    else:
        print("No valuations could be calculated.")
    
    print("\nAnalysis complete.")

if __name__ == "__main__":
    import os
    main()

INVESTMENT BANKING VALUATION TOOLKIT (with Full Sensitivity)

Enter company ticker symbol (e.g., AAPL, MSFT, JPM): GE

Analyzing GE...
Company: GE Aerospace
Sector: Industrials
Current Price: $281.16

--- DCF Valuation ---
Range: $107.41 - $129.26
DCF sensitivity table generated.
Heatmap saved as dcf_sensitivity_heatmap.png

--- Comparable Company Analysis ---
Found 5 peers
Range: $200.04 - $415.85
Percentile valuations computed.

--- Precedent Transaction Analysis ---
Range: $58.08 - $176.97

--- LBO Analysis ---
Range: $43.03 - $64.55
LBO sensitivity table generated.

--- Sum-of-the-Parts Analysis ---
Range: $99.97 - $149.95

Football field chart saved as football_field.png
Excel report saved as Valuation_Report.xlsx

Analysis complete.
